# 02 - De RDDs a DataFrames y Catalyst Optimizer

### Antipatrón de RDDs en Python
Los RDDs sufren sobrecarga de serialización entre la JVM y Python (Py4J). Los DataFrames usan memoria binaria fuera del recolector de basura (Project Tungsten) y optimizan consultas automáticamente mediante el motor **Catalyst**.


In [1]:
import sys
sys.path.append("..")
from src.config import get_spark_session
from src.etl.olympics_pipeline import DEPORTISTAS_SCHEMA
import pyspark.sql.functions as F

spark = get_spark_session("02_DataFrames")
df = spark.read.schema(DEPORTISTAS_SCHEMA).option("header", "true").csv("../data/raw/deportista.csv")
df.printSchema()
df.show(5)


root
 |-- deportista_id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- genero: integer (nullable = true)
 |-- edad: integer (nullable = true)
 |-- altura: double (nullable = true)
 |-- peso: double (nullable = true)
 |-- equipo_id: integer (nullable = true)

+-------------+--------------------+------+----+------+----+---------+
|deportista_id|              nombre|genero|edad|altura|peso|equipo_id|
+-------------+--------------------+------+----+------+----+---------+
|            1|           A Dijiang|     1|  24| 180.0|80.0|        1|
|            2|            A Lamusi|     1|  23| 170.0|60.0|        1|
|            3| Gunnar Nielsen Aaby|     1|  24| 175.0|72.0|        2|
|            4|Edgar Lindenau Aabye|     1|  34| 182.0|85.0|        2|
|            5|Christine Jacoba ...|     2|  21| 185.0|82.0|        3|
+-------------+--------------------+------+----+------+----+---------+
only showing top 5 rows



### Inspección del Plan Físico (`explain`)
Observa cómo Catalyst elimina columnas y filtra antes de cargar datos:


In [2]:
df_plan = df.filter(F.col("edad") > 20).select("nombre", "peso")
df_plan.explain(mode="formatted")


== Physical Plan ==
* Project (3)
+- * Filter (2)
   +- Scan csv  (1)


(1) Scan csv 
Output [3]: [nombre#1, edad#3, peso#5]
Batched: false
Location: InMemoryFileIndex [file:/c:/Users/Ruben/Desktop/Ciencia de datos/curso_spark/data/raw/deportista.csv]
PushedFilters: [IsNotNull(edad), GreaterThan(edad,20)]
ReadSchema: struct<nombre:string,edad:int,peso:double>

(2) Filter [codegen id : 1]
Input [3]: [nombre#1, edad#3, peso#5]
Condition : (isnotnull(edad#3) AND (edad#3 > 20))

(3) Project [codegen id : 1]
Output [2]: [nombre#1, peso#5]
Input [3]: [nombre#1, edad#3, peso#5]




### Ejercicio Práctico 2
Calcula la columna `peso_lb` multiplicando el peso por 2.20462 redondeado a 1 decimal.


In [3]:
# Solución validada:
df_calc = df.filter(F.col("peso").isNotNull()).withColumn("peso_lb", F.round(F.col("peso") * 2.20462, 1))
df_calc.select("nombre", "peso", "peso_lb").show(5)


+--------------------+----+-------+
|              nombre|peso|peso_lb|
+--------------------+----+-------+
|           A Dijiang|80.0|  176.4|
|            A Lamusi|60.0|  132.3|
| Gunnar Nielsen Aaby|72.0|  158.7|
|Edgar Lindenau Aabye|85.0|  187.4|
|Christine Jacoba ...|82.0|  180.8|
+--------------------+----+-------+
only showing top 5 rows

